# 4-bit Quantization for downstream QDoRA
Sean Browning

In [ ]:
from dotenv import load_dotenv
load_dotenv()
# Shim because nvidia container is CUDA13.1, but we have 13.0 installed
# and the two are mutually compatible, but we need to point to the right
# shared libraries for BnB
# BNB_CUDA_VERSION = 130

from pathlib import Path
from datetime import datetime

import torch
from safetensors.torch import save_file
import bitsandbytes as bnb
from conch.open_clip_custom.coca_model import CoCa, resize_pos_embed

CHECKPOINT_DIR = Path("../checkpoints/CONCH")
DEVICE = "cuda"

# Saved CoCa model config from file
CONCH_CONFIG = {
    "embed_dim": 512,
    "embed_dim_caption": 768,
    "vision_cfg": {
        "image_size": 448,
        "patch_size": 16,
        "attentional_pool_caption": True,
        "attentional_pool_contrast": True,
        "attn_pooler_heads": 8,
        "n_queries_contrast": 1,
        "n_queries_caption": 256,
        "output_tokens": True
    },
    "text_cfg": {
        "context_length": 128,
        "vocab_size": 32007,
        "width": 768,
        "heads": 12,
        "layers": 12,
        "embed_cls": True,
        "output_tokens": True
    },
    "multimodal_cfg": {
        "context_length": 128,
        "vocab_size": 32007,
        "width": 768,
        "heads": 12,
        "layers": 12
    }
}

# OpenAI color normalization values (from open_clip)
OPENAI_DATASET_MEAN = (0.48145466, 0.4578275, 0.40821073)
OPENAI_DATASET_STD = (0.26862954, 0.26130258, 0.27577711)

In [ ]:
def quantize_linear_to_4bit(model: torch.nn.Module, device: str = DEVICE):
    """
    Recursively quantize layers to 4-bit
    """
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            # Create a 4-bit Linear layer with NF4 quantization
            quant_layer = bnb.nn.Linear4bit(
                module.in_features,
                module.out_features,
                bias=module.bias is not None,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
            
            # Copy weights and bias
            quant_layer.weight.data = module.weight.data
            if module.bias is not None:
                quant_layer.bias.data = module.bias.data
                
            # Replace the layer in the model
            setattr(model, name, quant_layer)
        else:
            # Recursively apply to submodules
            quantize_linear_to_4bit(module, device)
            
    return model.to(device)

In [ ]:
# Construct Model
model = CoCa(**CONCH_CONFIG).to(DEVICE)

# Load in weights
weights = torch.load(CHECKPOINT_DIR / "pytorch_model.bin", map_location=DEVICE)
resize_pos_embed(weights, model)
model.load_state_dict(weights, strict=False)

model.visual.image_mean = OPENAI_DATASET_MEAN
model.visual.image_std = OPENAI_DATASET_STD

# Quantize to 4-bit
quantized_model = quantize_linear_to_4bit(model)

# Dump Quantized weights
save_file(
    quantized_model.state_dict(),
    CHECKPOINT_DIR / "conch_nf4.safetensors",
    metadata={
        "quant_type": "nf4",
        "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
)